# `agente_cz_A2A` · contrato A2A desde otro agente

Este notebook recorre lo que hace un agente supervisor para usar a `agente_cz_A2A`:

1. Descubrir el `assistant_id` del grafo en el servidor.
2. Leer su Agent Card.
3. Enviarle una tarea por `message/send`.
4. Reutilizar el `contextId` para conservar la memoria.
5. Delegar desde una función que cualquier supervisor puede importar.

Antes de ejecutar: `.venv\Scripts\langgraph.exe dev --port 2029` con el `.env` configurado.

In [1]:
import json
import os
import uuid

import httpx

# URL local o la del túnel de Cloudflare ($env:A2A_CZ_A2A_URL).
BASE_URL = os.environ.get("A2A_CZ_A2A_URL") or "http://127.0.0.1:2029"
GRAFO = "agente_cz_A2A"
http = httpx.AsyncClient(timeout=180, trust_env=False)
print("Servidor:", BASE_URL, "· grafo:", GRAFO)

Servidor: http://127.0.0.1:2029 · grafo: agente_cz_A2A


## 1. Descubrir el asistente

El servidor asigna un UUID a cada grafo; se busca por `graph_id`.

In [2]:
assert (await http.get(f"{BASE_URL}/ok")).status_code == 200, "Levanta langgraph dev --port 2029"

asistentes = (await http.post(f"{BASE_URL}/assistants/search", json={"graph_id": GRAFO})).json()
ASSISTANT_ID = asistentes[0]["assistant_id"]
print("assistant_id:", ASSISTANT_ID)

assistant_id: a4129127-cf48-5436-b0ca-71a49c148e3d


## 2. Leer la Agent Card

La tarjeta es el contrato: nombre, descripción, skills y la `url` del endpoint A2A. Si el agente se
publica por túnel, el cliente remoto debe reescribir esa URL antes de enviar.

In [3]:
tarjeta = (await http.get(f"{BASE_URL}/.well-known/agent-card.json", params={"assistant_id": ASSISTANT_ID})).json()
print(json.dumps({k: tarjeta.get(k) for k in ("name", "description", "url")}, indent=2, ensure_ascii=False))
print("skills:", [s.get("name") for s in tarjeta.get("skills", [])])
assert tarjeta["name"] == GRAFO

{
  "name": "agente_cz_A2A",
  "description": "agente_cz_A2A assistant",
  "url": "http://127.0.0.1:2029/a2a/a4129127-cf48-5436-b0ca-71a49c148e3d"
}
skills: ['agente_cz_A2A Capabilities']


## 3. Enviar una tarea (`message/send`)

La tarea va como texto. El `contextId` es el `thread_id` del servidor: con él el agente recuerda los
turnos anteriores.

In [4]:
def texto_de(resultado: dict) -> str:
    """Extrae el texto de un resultado A2A (status.message o history)."""
    mensaje = (resultado.get("status") or {}).get("message") or {}
    partes = [p.get("text", "") for p in mensaje.get("parts", []) if isinstance(p, dict)]
    if not any(partes):
        for turno in reversed(resultado.get("history") or []):
            if str(turno.get("role", "")).lower() in {"agent", "assistant"}:
                partes = [p.get("text", "") for p in turno.get("parts", []) if isinstance(p, dict)]
                break
    return "\n".join(p for p in partes if p).strip()


async def invocar(texto: str, context_id: str | None = None) -> tuple[str, str]:
    """Envía una tarea al agente y devuelve (respuesta, context_id)."""
    contexto = context_id or str(uuid.uuid4())
    cuerpo = {
        "jsonrpc": "2.0",
        "id": uuid.uuid4().hex,
        "method": "message/send",
        "params": {"message": {
            "role": "user",
            "messageId": uuid.uuid4().hex,
            "contextId": contexto,
            "parts": [{"kind": "text", "text": texto}],
        }},
    }
    datos = (await http.post(f"{BASE_URL}/a2a/{ASSISTANT_ID}", json=cuerpo)).json()
    if "error" in datos:
        raise RuntimeError(datos["error"])
    resultado = datos["result"]
    assert resultado["status"]["state"] == "completed", resultado["status"]
    return texto_de(resultado), resultado.get("contextId", contexto)


respuesta, CONTEXT_ID = await invocar("¿Cuánto vale AAPL en el mercado US y qué debería revisar antes de invertir?")
print(respuesta)

- **Datos**: AAPL | Precio: 342.10 USD (331.152,80 CLP) | Fecha de corte: 2026-11-07 (Datos simulados del taller). Variación diaria: +1,24%. Volumen: 54.300.000 acciones. Rango 52 semanas: 272.395,20 - 344.511,20 CLP (Distancia al máximo: 3,88%).
- **Fundamentales**: PER: 31,4 | Deuda/Patrimonio: 1,87 | Margen bruto: 46,2% | Margen operativo: 31,5% | Margen neto: 26,3% | Fecha de corte: 2026-11-07 (Datos simulados del taller).
- **Lectura**: AAPL cotiza cerca de su máximo de 52 semanas, respaldada por márgenes sólidos (operativo superior al 30% y neto sobre el 26%). No obstante, el ratio de endeudamiento de 1,87 y un PER de 31,4 exigen cautela frente a valoraciones elevadas.
- **Tesis**: La entrada se justifica por la alta rentabilidad sobre ventas y la posición dominante de mercado, siempre que el inversionista asuma el riesgo de pagar un múltiplo exigente.
- **Riesgos**: Una compresión del PER por menores crecimientos esperados o un incremento en los costos de deuda que afecte el rat

## 4. Memoria: reutilizar el `contextId`

El segundo turno puede referirse al primero sin repetir el contexto.

In [5]:
respuesta_2, CONTEXT_ID = await invocar(
    "Con esos datos, dame la tesis para un perfil moderado a 12 meses.", context_id=CONTEXT_ID
)
print("contextId reutilizado:", CONTEXT_ID)
print(respuesta_2)

contextId reutilizado: 87009a83-e3c8-47e5-9777-801179ea6de4
- **Datos**: AAPL | Precio: 342.10 USD (331.152,80 CLP) | Fecha de corte: 2026-11-07 (Datos simulados del taller).
- **Fundamentales**: PER: 31,4 | Deuda/Patrimonio: 1,87 | Margen neto: 26,3% | Fecha de corte: 2026-11-07 (Datos simulados del taller).
- **Tesis**: Para un perfil moderado y un horizonte de 12 meses, la tesis se apoya en la resiliencia operativa de Apple (márgenes sólidos superiores al 26% neto). Sin embargo, al cotizar a un PER de 31,4 y cerca de sus máximos de 52 semanas (a solo un 3,88%), el margen de seguridad es acotado. La estrategia se basa en capturar valor por estabilidad de flujo de caja, no por crecimiento explosivo a corto plazo.
- **Riesgos**: Compresión de múltiplos por desaceleración en ventas de hardware o alzas sostenidas en tasas que castiguen valoraciones altas con un endeudamiento de 1,87 veces patrimonio. Una caída bajo el soporte de las 52 semanas inferior a 280.000 CLP invalidaría el horizo

## 5. Delegación desde un agente supervisor

Un supervisor solo necesita esta función: resolver el `assistant_id` una vez y reenviar la
respuesta tal cual. Si delegas en varios agentes, usa un `contextId` por agente.

In [6]:
async def delegar_a_agente_cz_a2a(tarea: str, context_id: str | None = None) -> str:
    respuesta, _ = await invocar(tarea, context_id)
    return respuesta


print(await delegar_a_agente_cz_a2a("Compara NVDA y MSFT en US para alguien con horizonte de 3 años."))
await http.aclose()

- **Datos NVDA**: NVDA, USD 182.40 (CLP 176.563,20), actualizado al 2026-11-07 (datos simulados del taller).
- **Fundamentales NVDA**: PER 48,6; Deuda/Patrimonio 0,16; Margen bruto 74,9%; Margen operativo 61,2%; Margen neto 55,0% (Fecha de corte: 2026-11-07, datos simulados).

- **Datos MSFT**: MSFT, USD 443,05 (CLP 428.872,40), actualizado al 2026-11-07 (datos simulados del taller).
- **Fundamentales MSFT**: PER 35,2; Deuda/Patrimonio 0,42; Margen bruto 69,8%; Margen operativo 44,6%; Margen neto 35,9% (Fecha de corte: 2026-11-07, datos simulados).

- **Lectura**: Ambas cotizan cerca de sus máximos de 52 semanas (NVDA a 6,75% y MSFT a 5,37% de su techo), lo que refleja fortaleza pero menor margen de seguridad por precio. NVDA muestra márgenes e intensidad de crecimiento excepcionales con un balance muy sólido (deuda/patrimonio de 0,16), aunque con un PER exigente de 48,6. MSFT presenta un perfil más diversificado y equilibrado, con un PER de 35,2 y una estructura financiera también san

## Notas

- Los datos de las herramientas son **simulados del taller**: el agente siempre cita la fecha de corte.
- El agente no ejecuta pagos ni inversiones; deriva lo tributario y lo normativo.
- Si usas `a2a-sdk` en lugar de `httpx`, fija `protocol_version = "0.3.0"` en la tarjeta antes de
  `create_client(...)`: `langgraph dev` anuncia 1.0 pero responde el JSON de A2A 0.3.